C:\Users\moham\AppData\Local\Temp\ipykernel_13696\180259638.py:4: DtypeWarning: Columns (23,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("bdnb_capvm_10k.csv")


In [ ]:
! pip install pandas faiss-cpu sentence-transformers transformers


   ---------------------------------------- 0.0/340.6 kB ? eta -:--:--
   --- ------------------------------------ 30.7/340.6 kB 1.3 MB/s eta 0:00:01
   --- ------------------------------------ 30.7/340.6 kB 1.3 MB/s eta 0:00:01
   -------- ------------------------------ 71.7/340.6 kB 653.6 kB/s eta 0:00:01
   ------------ ------------------------- 112.6/340.6 kB 726.2 kB/s eta 0:00:01
   --------------- ---------------------- 143.4/340.6 kB 708.1 kB/s eta 0:00:01
   ------------------- ------------------ 174.1/340.6 kB 748.1 kB/s eta 0:00:01
   ------------------------- ------------ 225.3/340.6 kB 724.0 kB/s eta 0:00:01
   -------------------------- ----------- 235.5/340.6 kB 719.7 kB/s eta 0:00:01
   ------------------------------ ------- 276.5/340.6 kB 710.0 kB/s eta 0:00:01
   ----------------------------------- -- 317.4/340.6 kB 701.4 kB/s eta 0:00:01
   -------------------------------------- 340.6/340.6 kB 704.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/10.

In [1]:
! pip install bitsandbytes

  Using cached bitsandbytes-0.45.5-py3-none-win_amd64.whl.metadata (5.1 kB)
Using cached bitsandbytes-0.45.5-py3-none-win_amd64.whl (75.4 MB)


In [2]:
! pip install langchain faiss-cpu transformers accelerate sentence-transformers

  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.6.0-py3-none-any.whl (354 kB)


In [4]:
! pip install -U langchain-community


  Using cached langchain_community-0.3.21-py3-none-any.whl.metadata (2.4 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.8.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached httpx_sse-0.4.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached marshmallow-3.26.1-py3-none-any.whl.metadata (7.3 kB)
Using cached langchain_community-0.3.21-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.0-py3-none-any.whl (7.8 kB)
Using cached pydantic_settings-2.8.1-py3-none-any.whl (30 kB)
Using cached marshmallow-3.26.1-py3-none-any.whl (50 kB)


In [ ]:
# 📘 RAG Pipeline avec LangChain + Flan-T5 (Local RAG performant sur CSV BDNB)

import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from langchain.docstore.document import Document
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# -------------------------
# 1. Chargement et prétraitement des données
# -------------------------
df = pd.read_csv("bdnb_capvm_500k.csv")
df = df[[
    "libelle_commune_insee",
    "code_departement_insee",
    "s_geom_groupe",
    "rpls_classe_ener_principale",
    "ffo_bat_annee_construction"
]].dropna().astype(str)

# -------------------------
# 2. Conversion en documents LangChain
# -------------------------
documents = [Document(page_content=str(row.to_dict())) for _, row in df.iterrows()]

# -------------------------
# 3. Embedding avec all-MiniLM-L6-v2 (HuggingFace)
# -------------------------
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# -------------------------
# 4. Chargement du modèle Flan-T5 (local, compatible CPU)
# -------------------------
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
gen_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

# -------------------------
# 5. Création du pipeline RAG complet
# -------------------------
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

# -------------------------
# 6. Boucle interactive dans le terminal
# -------------------------


C:\Users\moham\AppData\Local\Temp\ipykernel_30468\1773967291.py:14: DtypeWarning: Columns (23,24,25,71,196,197,198,200,201) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("bdnb_capvm_500k.csv")
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu
